# Voice activity detection (VAD): hands-on

Companion to the manual's VAD section. You will:

1. load 60 s of a Thai meeting, recorded twice: on a close-talk microphone and on a microphone 3 m away,
2. run Silero, TEN VAD and WebRTC on it and see where each finds speech,
3. watch Silero miss the distant recording, then fix it by raising the level,
4. score every run against a human reference, and cut the speech into WAV files for speech-to-text.

Audio: [LOTUSDIS](https://huggingface.co/datasets/nectec/LOTUSDIS) (NECTEC), CC BY-SA 4.0. See the README for the exact source.

## Setup

**Locally:** clone the repository and run `uv sync`, then `uv run jupyter lab` (see the README). Everything is already installed, so this cell does nothing.

**On Colab:** it installs the three VAD libraries at the versions the manual was tested with, plus `libc++1`, which TEN VAD's Linux library needs. Colab already has PyTorch, NumPy, pandas and Matplotlib.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !apt-get -qq install -y libc++1
    %pip install -q silero-vad==6.2.2 onnxruntime ten-vad==1.0.6.9 webrtcvad-wheels==2.0.14.post1

Get the sample files. From a local clone, they are already here. In Colab, upload the `data/` folder with the Files panel on the left: while this repository is private, Colab cannot download from it. Once it is public, set `REPO_RAW` to its raw URL instead.

In [ ]:
from pathlib import Path
import urllib.request

REPO_RAW = ""  # e.g. "https://raw.githubusercontent.com/<owner>/thai-speech-notebooks/main/"
DATA = Path("data/lotusdis-S002_T062")

for name in ["close.wav", "distant.wav", "reference.csv"]:
    path = DATA / name

    if not path.exists():
        assert REPO_RAW, f"{path} is missing: upload data/ (see the cell above) or set REPO_RAW"
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + str(path), path)

print(sorted(p.name for p in DATA.iterdir()))

## Settings

Every number the notebook uses, in one place. Change one here, then rerun the cells below; the text quotes the defaults.

In [ ]:
import numpy as np

# Audio format all three models expect
SAMPLE_RATE = 16_000        # Hz
INT16 = np.iinfo(np.int16)  # 16-bit sample range: -32768..32767
INT16_SCALE = -INT16.min    # 32768: dividing a 16-bit sample by this gives -1.0..1.0

# Model settings: the manual's Getting started defaults
THRESHOLD = 0.5             # Silero and TEN VAD: score a frame needs to count as speech
SILERO_CHUNK = 512          # samples Silero scores at a time at 16 kHz (32 ms)
TEN_HOP = 256               # samples per TEN VAD frame (16 ms); 160 (10 ms) also works
WEBRTC_MODE = 3             # aggressiveness 0 to 3: 3 calls the least audio speech
WEBRTC_FRAME_MS = 30        # WebRTC accepts only 10, 20 or 30

# Level normalization
TARGET_DBFS = -20           # RMS level the manual raises distant audio to

# Scoring
MISS_WEIGHT = 0.75          # DCF = 0.75 x FN rate + 0.25 x FP rate: a missed second costs 3 false ones (NIST)
SCORE_STEP_S = 0.01         # time grid of the rough frame-by-frame score, in seconds
SEGMENTS_TO_PLAY = 3        # segments to listen to in step 6

## 1. Check the audio format

All three models want `SAMPLE_RATE` (16 kHz), mono, 16-bit WAV. The sample files already are. For your own audio, convert first:

```bash
ffmpeg -i input.mp3 -ac 1 -ar 16000 audio_16k.wav
```

In [ ]:
import wave

import pandas as pd
from IPython.display import Audio, display


def read_wav(path: Path) -> np.ndarray:
    """Read a mono 16-bit WAV recorded at SAMPLE_RATE.

    Args:
        path: The WAV file.

    Returns:
        The 16-bit samples.
    """
    with wave.open(str(path), "rb") as f:
        assert f.getnchannels() == 1 and f.getsampwidth() == 2, "convert to 16-bit mono first"
        assert f.getframerate() == SAMPLE_RATE, "convert to 16 kHz first"

        return np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16)


def cut(pcm: np.ndarray, start: float, end: float) -> np.ndarray:
    """Cut a stretch out of a recording.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        start: Start of the stretch, in seconds.
        end: End of the stretch, in seconds.

    Returns:
        The samples from start to end.
    """
    return pcm[int(start * SAMPLE_RATE):int(end * SAMPLE_RATE)]


close = read_wav(DATA / "close.wav")
distant = read_wav(DATA / "distant.wav")

reference = list(pd.read_csv(DATA / "reference.csv").itertuples(index=False, name=None))

print(f"{len(close) / SAMPLE_RATE:.0f} s; the reference has {len(reference)} speech segments")

Listen to both. Same people, same moment: the distant one is much quieter and more muffled.

In [ ]:
display(Audio(close, rate=SAMPLE_RATE))
display(Audio(distant, rate=SAMPLE_RATE))

## 2. Run the three models

Each function returns a `VadResult`: the speech segments as `(start, end)` pairs in seconds, plus the time and score of every frame for plotting. They follow the manual's Getting started snippets: Silero with its library defaults, TEN VAD at `THRESHOLD` (0.5) with `TEN_HOP` (256) samples per frame, WebRTC at `WEBRTC_MODE` (3) with `WEBRTC_FRAME_MS` (30 ms) frames. TEN VAD and WebRTC have no smoothing, so their segments come straight from the frame flags.

In [ ]:
from typing import NamedTuple

import torch
import webrtcvad
from silero_vad import load_silero_vad, get_speech_timestamps
from ten_vad import TenVad

Segments = list[tuple[float, float]]  # (start, end) of each speech segment, in seconds

SILERO_MODEL = load_silero_vad()


class VadResult(NamedTuple):
    """What every model function below returns."""

    segments: Segments  # what the model called speech
    times: np.ndarray  # start of each frame, in seconds
    scores: np.ndarray  # each frame's speech probability, or 0/1 flag for WebRTC


def frame_starts(pcm: np.ndarray | torch.Tensor, hop: int) -> range:
    """Find where each whole frame starts.

    Args:
        pcm: The samples.
        hop: Samples per frame.

    Returns:
        The sample index where each whole frame starts.
    """
    return range(0, len(pcm) - hop + 1, hop)


def frame_times(n_frames: int, hop: int) -> np.ndarray:
    """Convert frame numbers to times.

    Args:
        n_frames: How many frames.
        hop: Samples per frame.

    Returns:
        The start time of each frame, in seconds.
    """
    return np.arange(n_frames) * hop / SAMPLE_RATE


def flags_to_segments(flags: list[bool] | np.ndarray, hop: int) -> Segments:
    """Turn per-frame speech flags into segments.

    Args:
        flags: One speech flag per frame.
        hop: Samples per frame.

    Returns:
        The (start, end) of each run of speech frames, in seconds.
    """
    edges = np.diff(np.r_[0, np.asarray(flags, int), 0])
    starts = np.flatnonzero(edges == 1)
    ends = np.flatnonzero(edges == -1)

    hop_s = hop / SAMPLE_RATE
    return [(a * hop_s, b * hop_s) for a, b in zip(starts, ends)]


def silero(pcm: np.ndarray, threshold: float = THRESHOLD) -> VadResult:
    """Run Silero VAD, with its own smoothing at the library defaults.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        threshold: Score a frame needs to count as speech.

    Returns:
        Silero's segments, plus every frame's speech probability.
    """
    wav = torch.from_numpy(pcm / INT16_SCALE).float()

    # Segments; time_resolution=3 keeps milliseconds (the default rounds to 0.1 s)
    found = get_speech_timestamps(
        wav, SILERO_MODEL, sampling_rate=SAMPLE_RATE, threshold=threshold,
        return_seconds=True, time_resolution=3,
    )
    segments = [(s["start"], s["end"]) for s in found]

    # Per-frame probabilities, for the plot, from a fresh model state
    SILERO_MODEL.reset_states()
    scores = np.array([
        SILERO_MODEL(wav[i:i + SILERO_CHUNK], SAMPLE_RATE).item()
        for i in frame_starts(wav, SILERO_CHUNK)
    ])

    return VadResult(segments, frame_times(len(scores), SILERO_CHUNK), scores)


def ten(pcm: np.ndarray, threshold: float = THRESHOLD) -> VadResult:
    """Run TEN VAD, frame by frame, with no smoothing.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        threshold: Probability above which a frame counts as speech.

    Returns:
        TEN VAD's segments, plus every frame's speech probability.
    """
    vad = TenVad(TEN_HOP, threshold)

    results = [vad.process(pcm[i:i + TEN_HOP]) for i in frame_starts(pcm, TEN_HOP)]
    scores = np.array([prob for prob, _ in results])
    flags = [flag for _, flag in results]

    return VadResult(flags_to_segments(flags, TEN_HOP), frame_times(len(scores), TEN_HOP), scores)


def webrtc(pcm: np.ndarray, mode: int = WEBRTC_MODE) -> VadResult:
    """Run WebRTC VAD, frame by frame, with no smoothing.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        mode: Aggressiveness, 0 to 3.

    Returns:
        WebRTC's segments, plus every frame's 0/1 speech flag.
    """
    vad = webrtcvad.Vad(mode)
    hop = SAMPLE_RATE * WEBRTC_FRAME_MS // 1000  # samples per frame

    flags = np.array([vad.is_speech(pcm[i:i + hop].tobytes(), SAMPLE_RATE) for i in frame_starts(pcm, hop)])

    return VadResult(flags_to_segments(flags, hop), frame_times(len(flags), hop), flags.astype(float))


MODELS = {"Silero": silero, "TEN VAD": ten, "WebRTC": webrtc}

A plot per model: grey is speech in the human reference, the line is the model's score, the dashed line is `THRESHOLD`, the blue bar under it is what the model called speech. Pass `reference=None` for audio without one.

In [ ]:
import matplotlib.pyplot as plt


def plot(pcm: np.ndarray, title: str, reference: Segments | None = reference) -> None:
    """Plot every model's score and segments on one recording.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        title: Title above the plots.
        reference: Human speech segments to shade in grey; None for audio without one.
    """
    fig, axes = plt.subplots(len(MODELS), 1, figsize=(12, 1.8 * len(MODELS)), sharex=True)

    for ax, (name, run_model) in zip(axes, MODELS.items()):
        result = run_model(pcm)

        # Grey: speech in the reference
        for s, e in reference or []:
            ax.axvspan(s, e, color="0.85", lw=0)

        # Line: the model's score per frame; dashed: the threshold
        ax.plot(result.times, result.scores, lw=0.8, color="k")
        ax.axhline(THRESHOLD, ls="--", lw=0.8, color="tab:orange")

        # Blue bar: what the model called speech
        for s, e in result.segments:
            ax.plot([s, e], [-0.12, -0.12], lw=5, color="tab:blue", solid_capstyle="butt")

        ax.set_ylim(-0.2, 1.05)
        ax.set_ylabel(name)

    axes[0].set_title(title)
    axes[-1].set_xlabel("seconds")
    plt.tight_layout()
    plt.show()


plot(close, "Close-talk microphone")

## 3. The same speech, 3 m away

Silero is the manual's default, but in the manual's tests it missed most of the speech on this 3 m microphone, while TEN VAD did not. Compare Silero's line with TEN VAD's here.

In [ ]:
plot(distant, "Distant microphone (3 m)")

## 4. Raise the level, then try again

The level is the root mean square (RMS) of the samples, in dBFS: 0 dBFS is the loudest a sample can be. The manual raises distant audio to `TARGET_DBFS` (−20 dBFS). Gain G = target − level, and every sample is multiplied by 10^(G/20). Peaks pushed past 0 dBFS are cut off (clipping): this prints how much.

This needs the whole recording first, so it cannot run on live audio. That is why the manual recommends TEN VAD for live distant-mic audio.

In [ ]:
def level_dbfs(pcm: np.ndarray) -> float:
    """Measure a recording's level.

    Args:
        pcm: 16-bit samples.

    Returns:
        The RMS level in dBFS; 0 is the loudest a sample can be.
    """
    x = pcm / INT16_SCALE
    return 20 * np.log10(np.sqrt(np.mean(x**2)))


level = level_dbfs(distant)
gain = TARGET_DBFS - level
factor = 10 ** (gain / 20)

scaled = distant * factor
louder = np.clip(np.round(scaled), INT16.min, INT16.max).astype(np.int16)

print(f"level {level:.1f} dBFS, gain {gain:+.1f} dB (every sample x {factor:.2f})")
print(f"clipped samples: {np.mean(np.abs(scaled) > INT16.max):.3%}")

display(Audio(louder, rate=SAMPLE_RATE))

In [ ]:
plot(louder, f"Distant microphone after {gain:+.1f} dB")

## 5. Score against the reference

FN rate: share of reference speech the model missed. FP rate: share of non-speech it called speech. DCF weighs them by `MISS_WEIGHT` (0.75 × FN rate + 0.25 × FP rate), so a missed second costs 3 times a false one; lower is better.

This is a rough, frame-by-frame score on 60 s. The manual's numbers use pyannote.metrics with a 200 ms collar on longer recordings, so expect them to differ.

In [ ]:
def score(segments: Segments, reference: Segments, duration: float) -> tuple[float, float, float]:
    """Score a model's segments against a reference, on a SCORE_STEP_S time grid.

    Args:
        segments: What the model called speech.
        reference: Speech in the human reference.
        duration: Length of the recording, in seconds.

    Returns:
        The FN rate, FP rate and DCF.
    """
    t = np.arange(0, duration, SCORE_STEP_S)

    def inside(segs: Segments) -> np.ndarray:
        if not segs:
            return np.zeros(len(t), bool)
        return np.any([(t >= s) & (t < e) for s, e in segs], axis=0)

    called = inside(segments)
    truth = inside(reference)

    fn = (~called & truth).sum() / truth.sum()
    fp = (called & ~truth).sum() / (~truth).sum()
    dcf = MISS_WEIGHT * fn + (1 - MISS_WEIGHT) * fp

    return fn, fp, dcf


rows = []

for audio_name, pcm in {"close": close, "distant": distant, "distant + gain": louder}.items():
    for name, run_model in MODELS.items():
        segments = run_model(pcm).segments
        fn, fp, dcf = score(segments, reference, len(pcm) / SAMPLE_RATE)
        rows.append({"audio": audio_name, "model": name, "FN rate": fn, "FP rate": fp, "DCF": dcf})

pd.DataFrame(rows).pivot(index="model", columns="audio", values="DCF").round(3)

## 6. Check by ear, then save segments for speech-to-text

Scores come from one reference. Before trusting a model on your own audio, listen: is each segment speech, starting and ending with the words? Here are the first `SEGMENTS_TO_PLAY` segments Silero found on the louder audio.

In [ ]:
segments = silero(louder).segments

for s, e in segments[:SEGMENTS_TO_PLAY]:
    print(f"{s:.2f}s -> {e:.2f}s")
    display(Audio(cut(louder, s, e), rate=SAMPLE_RATE))

Write each segment to its own WAV, the input the speech-to-text section starts from.

In [ ]:
out = Path("segments")
out.mkdir(exist_ok=True)

for i, (s, e) in enumerate(segments):
    with wave.open(str(out / f"segment_{i:04d}.wav"), "wb") as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(SAMPLE_RATE)
        f.writeframes(cut(louder, s, e).tobytes())

print(len(segments), "segments written to", out)

## Try next

- Lower Silero's threshold on the distant audio: `plot` uses `THRESHOLD`, try `silero(distant, threshold=0.1)`. Does any threshold fix it without the gain?
- WebRTC on the distant audio at mode 0 instead of 3: `webrtc(distant, mode=0)`.
- Run your own recording: convert it with the ffmpeg command in step 1, `read_wav()` it, and `plot(pcm, title, reference=None)`.

## Exercises

Each exercise is a function to finish: fill in the code under `###### Your code here ######`, following the step banners, and `return` the answer. Then run the check cell below it: it prints "passed", or stops at an `AssertionError` whose message is a hint. Run every cell above first; the exercises use its functions and audio.

In [ ]:
# Exercise settings
MIN_SILENCE_S = 0.3   # merge segments whose gap is shorter than this (the manual's example: 300 ms)
MIN_SPEECH_S = 0.25   # drop segments shorter than this (Silero's min_speech_duration_ms default: 250)
PAD_S = 0.03          # add this to both ends of every segment (Silero's speech_pad_ms default: 30)
SWEEP = np.round(np.arange(0.05, 1.0, 0.05), 2)  # thresholds to try in exercise 6: 0.05, 0.10, ... 0.95
ANSWER_TOLERANCE = 0.01                          # how close a returned number must be


def assert_close(mine: float | None, expected: float, hint: str) -> None:
    """Check a returned number without giving the answer away.

    Args:
        mine: The number your function returned.
        expected: The right answer.
        hint: What to print if they differ.
    """
    assert mine is not None, "your function returned nothing: finish it and `return` the answer"
    assert abs(mine - expected) <= ANSWER_TOLERANCE, hint

### Exercise 1: frames

How long is a recording in seconds, and how many whole frames of a given length fit in it? Write both functions; the check runs them on the close-talk clip with `WEBRTC_FRAME_MS` (30 ms) frames.

In [ ]:
def clip_seconds(pcm: np.ndarray) -> float:
    """Measure a recording's length.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.

    Returns:
        The length in seconds.
    """
    ###### Your code here ######


def count_frames(pcm: np.ndarray, frame_ms: int) -> int:
    """Count the whole frames that fit in a recording.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        frame_ms: Frame length, in milliseconds.

    Returns:
        How many whole frames fit; a part frame at the end does not count.
    """
    ###### Your code here ######
    ###### 1. samples per frame ######

    ###### 2. whole frames that fit (round down) ######

In [ ]:
assert_close(clip_seconds(close), len(close) / SAMPLE_RATE, "length = number of samples / samples per second")
assert count_frames(close, WEBRTC_FRAME_MS) == len(webrtc(close).times), "count only whole frames: round down"

print("Exercise 1 passed")

### Exercise 2: score a model by hand

A 10 s recording: speech from 0 to 4 s and from 6 to 10 s. A model called speech from 1 to 4 s and from 5 to 10 s. Work out its FN rate, FP rate and DCF on paper (see step 5), then return the three numbers. Fractions are fine, e.g. `3 / 8`.

In [ ]:
toy_reference = [(0, 4), (6, 10)]
toy_decision = [(1, 4), (5, 10)]
TOY_DURATION_S = 10


def toy_rates() -> tuple[float, float, float]:
    """Give your hand-worked scores for the toy example above.

    Returns:
        The FN rate, FP rate and DCF, as numbers you worked out on paper.
    """
    ###### Your code here ######
    ###### 1. FN rate: missed speech / reference speech ######

    ###### 2. FP rate: false speech / reference non-speech ######

    ###### 3. DCF ######

In [ ]:
fn, fp, dcf = score(toy_decision, toy_reference, TOY_DURATION_S)
answer = toy_rates()
assert answer is not None, "toy_rates returned nothing: `return fn_rate, fp_rate, dcf`"
my_fn_rate, my_fp_rate, my_dcf = answer

assert_close(my_fn_rate, fn, "FN rate = seconds of speech the model missed / seconds of speech in the reference")
assert_close(my_fp_rate, fp, "FP rate = seconds the model wrongly called speech / seconds of non-speech in the reference")
assert_close(my_dcf, dcf, "DCF = MISS_WEIGHT x FN rate + (1 - MISS_WEIGHT) x FP rate")

print("Exercise 2 passed")

### Exercise 3: the model that always says speech

Before computing anything: what DCF does a "model" get if it calls the whole close-talk clip speech? Return your prediction; the check then computes it. This is why the manual uses DCF instead of F1: F1 would rate this useless model highly.

In [ ]:
def predict_always_speech_dcf() -> float:
    """Predict the DCF of a model that calls everything speech.

    Returns:
        Your predicted DCF, reasoned out without running anything.
    """
    ###### Your code here ######
    ###### 1. its FN rate: how much speech does it miss? ######

    ###### 2. its FP rate: how much non-speech does it call speech? ######

    ###### 3. DCF ######

In [ ]:
everything = [(0, len(close) / SAMPLE_RATE)]
fn, fp, dcf = score(everything, reference, len(close) / SAMPLE_RATE)

assert_close(predict_always_speech_dcf(), dcf, "it misses no speech (FN rate 0) but calls every second of non-speech speech (FP rate 1)")

print(f"Exercise 3 passed: FN rate {fn:.2f}, FP rate {fp:.2f}, DCF {dcf:.2f}")

### Exercise 4: fill short pauses

TEN VAD and WebRTC have no smoothing: the manual says to write it yourself. Write `merge_short_gaps`: join neighbouring segments whose gap is shorter than `min_gap_s`. Segments arrive sorted by start time; overlapping ones merge too.

In [ ]:
def merge_short_gaps(segments: Segments, min_gap_s: float) -> Segments:
    """Fill short pauses: join neighbouring segments whose gap is shorter than min_gap_s.

    Args:
        segments: (start, end) pairs in seconds, sorted by start.
        min_gap_s: Gaps shorter than this are filled. Overlapping segments merge too.

    Returns:
        The merged (start, end) pairs.
    """
    ###### Your code here ######
    ###### 1. walk the segments in order, keeping a list of merged ones ######

    ###### 2. if the gap to the last merged segment is short, extend it; otherwise start a new one ######

In [ ]:
assert merge_short_gaps([], 0.3) == [], "no segments in, no segments out"
assert merge_short_gaps([(0, 1)], 0.3) == [(0, 1)], "a single segment stays as it is"
assert merge_short_gaps([(0, 1), (1.2, 2), (3, 4)], 0.3) == [(0, 2), (3, 4)], "a 0.2 s gap is under 0.3 s, so merge; a 1 s gap is not"
assert merge_short_gaps([(0, 1), (1.1, 2), (2.1, 3)], 0.2) == [(0, 3)], "merging can chain across several gaps"
assert merge_short_gaps([(0, 2), (1.5, 3)], 0) == [(0, 3)], "overlapping segments merge, keeping the later end"

ten_segments = ten(close).segments
merged = merge_short_gaps(ten_segments, MIN_SILENCE_S)
gaps = [after[0] - before[1] for before, after in zip(merged, merged[1:])]
assert all(gap >= MIN_SILENCE_S for gap in gaps), "after merging, no gap should be shorter than MIN_SILENCE_S"

print(f"Exercise 4 passed: TEN VAD's {len(ten_segments)} segments became {len(merged)}")

### Exercise 5: drop short bursts, add padding

Write the other two smoothing steps. `drop_short` removes segments shorter than `min_len_s` (a cough, a click). `pad` widens every segment by `pad_s` on both ends, without going below 0 or past `duration`. `smooth` then runs all three in Silero's order.

In [ ]:
def drop_short(segments: Segments, min_len_s: float) -> Segments:
    """Drop short bursts, such as a cough or a click.

    Args:
        segments: (start, end) pairs in seconds.
        min_len_s: Segments shorter than this are dropped.

    Returns:
        The segments at least min_len_s long.
    """
    ###### Your code here ######


def pad(segments: Segments, pad_s: float, duration: float) -> Segments:
    """Widen every segment on both ends.

    Args:
        segments: (start, end) pairs in seconds.
        pad_s: Seconds to add before the start and after the end.
        duration: Length of the recording; no segment may go past it, or before 0.

    Returns:
        The padded (start, end) pairs.
    """
    ###### Your code here ######


def smooth(segments: Segments, duration: float) -> Segments:
    """Smooth segments in Silero's order: fill short pauses, drop short bursts, then pad.

    Args:
        segments: (start, end) pairs in seconds, sorted by start.
        duration: Length of the recording, in seconds.

    Returns:
        The smoothed (start, end) pairs.
    """
    merged = merge_short_gaps(segments, MIN_SILENCE_S)
    kept = drop_short(merged, MIN_SPEECH_S)
    return pad(kept, PAD_S, duration)

In [ ]:
assert drop_short([(0, 0.1), (1, 2)], 0.25) == [(1, 2)], "0.1 s is shorter than 0.25 s, so it goes"
assert drop_short([(0, 0.25)], 0.25) == [(0, 0.25)], "exactly min_len_s long stays"
assert pad([(1, 2), (5, 5.5)], 0.5, 5.8) == [(0.5, 2.5), (4.5, 5.8)], "pad both ends, but never past duration"
assert pad([(0.2, 1)], 0.5, 10) == [(0, 1.5)], "never before 0"

duration = len(distant) / SAMPLE_RATE
raw = ten(distant).segments
smoothed = smooth(raw, duration)
assert all(0 <= s < e <= duration for s, e in smoothed), "every segment must stay inside the recording"
assert all(e - s >= MIN_SPEECH_S for s, e in smoothed), "no segment should be shorter than MIN_SPEECH_S"

print("Exercise 5 passed")
print(f"TEN VAD on the distant clip, DCF without smoothing: {score(raw, reference, duration)[2]:.3f}")
print(f"TEN VAD on the distant clip, DCF with smoothing:    {score(smoothed, reference, duration)[2]:.3f}")

Did smoothing help TEN VAD here? Which of the three steps do you think mattered most, and why? Try changing `MIN_SILENCE_S`, `MIN_SPEECH_S` or `PAD_S` and rerun.

### Exercise 6: find Silero's best threshold

The manual reports each model at the threshold with its lowest DCF. Do the same for Silero on the louder distant audio: write `sweep_thresholds` to score Silero at every threshold in `SWEEP`, and `best_threshold` to pick the lowest DCF. The check runs Silero 19 times, so it may take a minute.

In [ ]:
def sweep_thresholds(pcm: np.ndarray, thresholds: np.ndarray) -> dict[float, float]:
    """Score Silero at every threshold.

    Args:
        pcm: 16-bit samples at SAMPLE_RATE.
        thresholds: Thresholds to try.

    Returns:
        Each threshold's DCF against `reference`.
    """
    ###### Your code here ######
    ###### 1. run silero(pcm, threshold) for each threshold ######

    ###### 2. score its segments against `reference` and keep the DCF ######


def best_threshold(dcf_by_threshold: dict[float, float]) -> float:
    """Pick the threshold with the lowest DCF.

    Args:
        dcf_by_threshold: Each threshold's DCF.

    Returns:
        The threshold with the lowest DCF.
    """
    ###### Your code here ######

In [ ]:
dcf_by_threshold = sweep_thresholds(louder, SWEEP)
assert dcf_by_threshold is not None, "sweep_thresholds returned nothing: return a dict of threshold -> DCF"
assert set(dcf_by_threshold) == set(SWEEP), "score every threshold in SWEEP, no more and no fewer"

best = best_threshold(dcf_by_threshold)
assert best == min(dcf_by_threshold, key=dcf_by_threshold.get), "the best threshold is the one with the lowest DCF"
assert dcf_by_threshold[best] <= dcf_by_threshold[THRESHOLD], "the best can never be worse than the default"

print(f"Exercise 6 passed: best threshold {best}, DCF {dcf_by_threshold[best]:.3f} "
      f"against {dcf_by_threshold[THRESHOLD]:.3f} at the default {THRESHOLD}")

The manual warns that a threshold tuned on one recording does worse on new audio. Would you ship the best threshold, or keep the default? What would you need to decide?